In [1]:
import tensorflow as tf
import numpy as np

print("TensorFlow Version:", tf.__version__)
print("Libraries Imported Successfully!")

TensorFlow Version: 2.21.0
Libraries Imported Successfully!


In [2]:
texts = [
    "I love this movie",
    "This film is excellent",
    "Amazing experience",
    "I enjoyed this product",
    "This is wonderful",
    "Very good service",
    "I hate this movie",
    "This film is terrible",
    "Very bad experience",
    "I dislike this product",
    "This is horrible",
    "Poor service"
]

labels = np.array([
    1, 1, 1, 1, 1, 1,
    0, 0, 0, 0, 0, 0
])

texts = np.array(texts).reshape(-1, 1)

print("Text Samples:", len(texts))
print("Labels:", labels)

Text Samples: 12
Labels: [1 1 1 1 1 1 0 0 0 0 0 0]


In [3]:
vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=1000,
    output_mode="int",
    output_sequence_length=10
)

vectorizer.adapt(texts)

print("Vocabulary:")
print(vectorizer.get_vocabulary())

Vocabulary:
['', '[UNK]', np.str_('this'), np.str_('is'), np.str_('i'), np.str_('very'), np.str_('service'), np.str_('product'), np.str_('movie'), np.str_('film'), np.str_('experience'), np.str_('wonderful'), np.str_('terrible'), np.str_('poor'), np.str_('love'), np.str_('horrible'), np.str_('hate'), np.str_('good'), np.str_('excellent'), np.str_('enjoyed'), np.str_('dislike'), np.str_('bad'), np.str_('amazing')]


In [4]:
# Convert text data into 1D format
texts = texts.reshape(-1)

# Prepare text vocabulary
vectorizer.adapt(texts)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(), dtype=tf.string),
    vectorizer,
    tf.keras.layers.Embedding(1000, 16),
    tf.keras.layers.LSTM(16),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ text_vectorization                   │ (None, 10)                  │               0 │
│ (TextVectorization)                  │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ embedding (Embedding)                │ (None, 10, 16)              │          16,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ (None, 16)                  │           2,112 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 1)                   │              17 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 18,129 (70.82 KB)

 Trainable params: 18,129 (70.82 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
# Convert text and labels into TensorFlow tensors
train_texts = tf.constant(texts.tolist(), dtype=tf.string)
train_labels = tf.constant(labels, dtype=tf.float32)

# Compile model
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Train model
history = model.fit(
    train_texts,
    train_labels,
    epochs=30,
    batch_size=4,
    verbose=1
)

print("Text Classification Training Completed!")

Epoch 1/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 8s 71ms/step - accuracy: 0.5000 - loss: 0.6930
Epoch 2/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 0.5000 - loss: 0.6929 
Epoch 3/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.6667 - loss: 0.6931
Epoch 4/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.5000 - loss: 0.6929
Epoch 5/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.4167 - loss: 0.6937    
Epoch 6/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.6667 - loss: 0.6925
Epoch 7/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.7500 - loss: 0.6924
Epoch 8/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.8333 - loss: 0.6920
Epoch 9/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.8333 - loss: 0.6919
Epoch 10/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.9167 - loss: 0.6915
Epoch 11/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.9167 - loss: 0.6912 
Epoch 12/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.9167 - loss: 0.

In [7]:
test_texts = tf.constant([
    "This movie is amazing",
    "This movie is very bad"
], dtype=tf.string)

predictions = model.predict(test_texts)

for text, prediction in zip(test_texts.numpy(), predictions):
    sentiment = "Positive" if prediction[0] >= 0.5 else "Negative"

    print(text.decode(), ":", sentiment)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 758ms/step
This movie is amazing : Positive
This movie is very bad : Negative
